<a href="https://colab.research.google.com/github/javirk/europa_surface/blob/revert_fixed/DEMO_apply_LineaMapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the demo of LineaMapper v1.1 and 2.0. In this jupyter notebook, you will be guided through the process of retrieving predictions with LineaMapper v1.0, v1.1 and v2.0. You can upload your own geotiff image or make use of the 'Region A' from the publication "Length, width, and relative age analysis of lineaments in the Galileo regional maps with LineaMapper" (Haslebacher et al., PSJ, 2025).

Have fun, Caroline.

By default, this script retrieves predictions on a region showing the southern leading hemisphere of Jupiter's moon Europa
* for LineaMapper v1.0
   * with 224 tilesize ('geosize')
* for LineaMapper v1.1
   * with 224 tilesize
   * with 112 tilesize
* for LineaMapper v2.0
   * with 112 tilesize

# Preparation

In [ ]:
IS_COLAB = False # execute this cell if you are NOT on Google colab, but on binder

In [38]:
IS_COLAB = True # execute this cell if you ARE on Google colab

In [39]:
import os
from pathlib import Path
from datetime import datetime
import time
from osgeo import ogr

In [40]:
# if you want, you can test if you have a GPU available with this cell
import torch
torch.cuda.is_available()
# we are searching for a GPU with
# self.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# later in the LineaMapper_vX_to_img.py scripts

True

In [41]:
# The below line clones the github repository to your local or remote machine
!git clone -b revert_fixed https://github.com/javirk/europa_surface.git

Cloning into 'europa_surface'...
remote: Enumerating objects: 2127, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 2127 (delta 107), reused 66 (delta 38), pack-reused 1940 (from 2)
Receiving objects: 100% (2127/2127), 5.26 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (1694/1694), done.


In [51]:
# we need to install the geojson module
!pip install geojson

We now clone the full github repository directly into Google Colab so that we can access every script. We change directory so that we are inside the cloned repository.

In [53]:
# we access the repository to import modules below
# also ONLY IF NOT ON BINDER:
if IS_COLAB == True:
    %cd europa_surface
    # and we need to install gdal in google Colab (following https://stackoverflow.com/questions/70275565/how-to-install-gdal-on-google-colab-fast)
    # this is for command line tools, which we'll use later
    !apt-get update
    !apt install gdal-bin


[Errno 2] No such file or directory: 'europa_surface'
/content/europa_surface/europa_surface
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,683 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,948 kB]
Get:12 http://archive.

If you want to run LineaMapper on your own geotiff, simply put/upload your geotiff to the folder './demo/v1_1' (and remove everything for which you do not want predictions). Here, we run LineaMapper on the geotiff in './demo/v1_1' of region A from the publication.

In [44]:
# define the input path
# note: basepath is used to define the savepath
basepath = Path('.')

# we get this from the github repository
source_path = basepath / './demo/v1_1'
# because the next line catches every file that is ending in '.tif', you can also upload your own geotiff file here
tifpaths = sorted(source_path.glob('*.tif'))

dt_string = datetime.now().strftime("%Y_%m_%d")

Next, we download the weights for LineaMapper version 1.0, 1.1 and 2.0 and put them into a subdirectory './ckpts' (checkpoints). If for any reason, the direct download form Mendeley fails (because the links were put in before they were activated), go to the Mendeley and download the weights manually and put them into a sub-directory called 'ckpts' in the directory where this notebook is located. The link to Mendeley is likely: https://data.mendeley.com/datasets/rjhsjrnxgv/1
You can also go to https://github.com/javirk/europa_surface and download the latest code. The weights correspond to versions as follows:
* LM1.0: Mask_R-CNN_pub2_run23_end_model.pt
* LM1.1: Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt
* LM2.0: bbox_vit_b_final.pt
* other files are other versions of LM2.0 (see paper)

In [45]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo (these work for sure)
# url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

url_LM1_0 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0/file_downloaded"
url_LM1_1 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/5b40d00e-dfdd-4803-857a-e87815988992/file_downloaded"
url_LM2_0 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/9d4f74fe-b794-4388-8c5e-416468277531/file_downloaded"

# # for testing, before Mendeley Data repo is published:
# url_LM1_0 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"
# url_LM1_1 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"
# url_LM2_0 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

for url, LMname in [(url_LM1_0, "Mask_R-CNN_pub2_run23_end_model.pt"), (url_LM1_1, "Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt"), (url_LM2_0, "bbox_vit_b_final.pt")]:
    print(url, LMname)
    # Download the file
    response = requests.get(url)
    response.raise_for_status()  # Ensure the request was successful

    # Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
    %mkdir ckpts
    with open(f"./ckpts/{LMname}", "wb") as file:
        file.write(response.content)



https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0/file_downloaded Mask_R-CNN_pub2_run23_end_model.pt
https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/5b40d00e-dfdd-4803-857a-e87815988992/file_downloaded Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt
mkdir: cannot create directory ‘ckpts’: File exists
https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/9d4f74fe-b794-4388-8c5e-416468277531/file_downloaded bbox_vit_b_final.pt
mkdir: cannot create directory ‘ckpts’: File exists


Below, we define a convenient straightforward function to execute LineaMapper_v1_to_img.py or LineaMapper_v2_to_img.py that we can call later for different versions. We are preparing a command-line that calls the script and passes on arguments. You can change these arguments and see what effect they have. Very briefly:


* geofile: full path with filename in TIFF format. The full path of the file for which predictions are seeked. Must be a GEOTIFF file.
* savedir: path to directory where the output is stored. there will be subdirectories automatically generated by the function.
* mask_threshold: threshold for float mask. The float mask that is output by the model gets converted to a binary mask using this threshold. Default is 0.5.
* iou_threshold: threshold for Intersection-Over-Union (IoU). The computed mask IoU is multiplied with the multiplication factor and then tested against the IoU threshold. Default is 0.5.
* multiplication_factor: factor by which computed mask Intersection-Over-Union (IoU) gets multiplied. This compensates for the moving window algorithm.
* del_pxs: If a boolean mask has an area lower than del_pxs, it gets deleted.
* class_scores: list with score thresholds for individual classes. If not given, defaults to 0.5 for each class. Classes are 1) bands 2) double ridges 3) ridge complexes 4) undifferentiated lineae.
* geosize: Choose a tile size that is fed to the network. This tile gets re-cast to 200x200 or 300x300. (minsize, maxsize)
* cut_size: Choose a size for cutting the input image into subimages. The image gets tiled up into smaller subimages, if it is bigger than cut_size.
* azimuth_diff_range: Choose a maximal difference between two azimuths so that they are still considered the same direction. Masks that fulfill the IoU criterion, but are not going into the same direction, are not merged.
* modelname: path to .pt file of the model.
* minsize/maxsize: Choose the minsize/maxsize parameter for the Mask R-CNN. This tile gets re-cast to (minsize, maxsize)
* sampath: .pt file of the SAM model, in the model_dict subdirectory.
* sam_modus: model architecture. vit_b or vit_t

You can explore all arguments in LineaMapper_v1_to_img.py.



In [46]:
import subprocess
# we define this routine only to get debugging help if anything goes wrong, and to see progress.
def run_command(command):
    try:
        # Run the command
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Capture standard output
        stdout = result.stdout
        # Capture standard error
        stderr = result.stderr

        # Print helpful debugging information
        print("Command executed successfully.")
        print("Standard Output:")
        print(stdout)

        if stderr:
            print("Standard Error:")
            print(stderr)

    except subprocess.CalledProcessError as e:
        # Capture error output in case of an error
        print("An error occurred while executing the command.")
        print("Standard Output:")
        print(e.stdout)
        print("Standard Error:")
        print(e.stderr)

    return


In [57]:
# We define a straightforward function to execute LineaMapper_v1.py or ..._v2.py that we can call later for different models
# this function automatically loops through all found tiff files
# it stores the output as a geojson and a shapefile, along with a text file with the parameters we used
def forward_LM(modelname, version, subset, geosize):
    # NOTE: the savepath does not yet need to exist
    savepath = basepath / 'LineaMapper_output' / (dt_string + '_RegionA_' + subset)
    # start full time
    full_time_start = time.time()

    for tifffile in tifpaths:
        tf = tifffile.stem
        print(tf)
        command = f'python LineaMapper_{version}_to_img.py --modelname={modelname} --geofile=' + str(source_path.joinpath(tf + '.tif')) + ' --savedir=' + str(savepath) + f' --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize={geosize}'
        print(command)
        run_command(command)

    # measure time and simply print
    timesum = time.time() - full_time_start
    print('this script took {:.2f} seconds to execute. Makes {:.2f} hours.'.format(timesum, timesum/3600))
    # simple test:
    # python LineaMapper_to_img.py --geofile=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/Europa_Mosaics_Equirectangular/E6ESCRATER01_GalileoSSI_Equi-cog.tif --savedir=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/LineaMapper_output/tests

    ######## convert to shapefiles
    # problem is that I do not have the exact filename, so I retrieve it simply afterwards
    # make shape_file folder
    os.makedirs(Path(savepath) / 'shape_files', exist_ok=True)

    geojfiles = sorted((Path(savepath) / 'json_files').glob('*.geojson'))
    for geojfile in geojfiles:
        # convert to shapefile as well
        command = 'ogr2ogr -nlt POLYGON -skipfailures {} {}'.format((savepath / 'shape_files').joinpath(geojfile.stem + '.shp'), (savepath / 'json_files').joinpath(geojfile.stem + '.geojson'))
        print(command)
        run_command(command)

    # check if it worked for all
    geoshpfiles = sorted((Path(savepath) / 'shape_files').glob('*.shp'))

    if len(geoshpfiles) == len(geojfiles):
        print('ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.')
    else:
        raise Warning('some geojson files lead to errors, it seems.')

    return

# Running LineaMapper

Now, we are already prepared to actually run LineaMapper on our input geotiff image(s). This can take up to 1 hour on a Google Colab CPU. Depending what computing power you have available, it can be much faster. In google colab, you can start up a GPU by going to 'runtime' > Change runtime type, and select a GPU. However, we have noticed that running the external script does not automatically recognise the GPU. If this happens to you, you might want to crop the geotiff input image for faster results.
Else, you can run the next cell to select a cropped version (800x800 px) of region A. This should take 4 minutes for one model.

In [48]:
# OPTIONAL: adapt tifpaths to take smaller image (for faster inference)
source_path = basepath / './demo/v1_1/800px' # we also need to adapt the source path
tifpaths = sorted(source_path.glob('*.tif'))

In [49]:
tifpaths

[PosixPath('demo/v1_1/800px/17ESREGMAP02_Bland2021_800px.tif')]

In [58]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Mask_R-CNN_pub2_run23_end_model.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_800px
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_pub2_run23_end_model.pt --geofile=demo/v1_1/800px/17ESREGMAP02_Bland2021_800px.tif --savedir=LineaMapper_output/2025_05_19_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224
Command executed successfully.
Standard Output:
17ESREGMAP02_Bland2021_800px
class scores: {1: np.float64(0.5), 2: np.float64(0.5), 3: np.float64(0.5), 4: np.float64(0.5)}
demo/v1_1/800px/17ESREGMAP02_Bland2021_800px.tif
minsize: 200, maxsize: 300
The input image was divided into 49 tiles.
0
1
2
3
4
5
concatenating now
referenced predictions
merging...
merged!
I will do a preview image.
For this image, I needed 220.08 seconds, for 49 tiles, so 4.49 per tile.
The time now is 2025-05-19 22:52:07.058467
For this image, I needed 221.20 seconds, for 49 tiles, so 4.51 per tile.
The time now is 2025-05-19 22:52:07.058823

Standard Error:
/u

In [ ]:
# for LineaMapper v1.1
#     on 224 geosize (tiles)
subset = 'LM1.1_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)
#     on 112 geosize
geosize = 112
subset = 'LM1.1_112'
forward_LM(modelname, version, subset, geosize)

In [ ]:
# for LineaMapper v2.0
#     on 112 geosize
# note that sampath and sammodus are the default. "./ckpts/bbox_vit_b_final.pt", 'vit_b'
subset = 'LM2.0_112' # identification string for savepath
geosize = 112
modelname = './ckpts/bbox_vit_b_final.pt'
version = 'v2' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

Now, you can download the output from generated folder "LineaMapper_output", pull them into a GIS application (such as open source QGIS), and inspect and compare the output. If you have used the demo image, you should also see a preview pdf image. (Please note that the preview feature is only working for images smaller than the cut size, because the display_preview routine is not currently updated for the use of subimages.)

If the Mendeley data download fails, try this:

In [ ]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo for Haslebacher et al. (2024)
url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

# Download the file
response = requests.get(url)
response.raise_for_status()  # Ensure the request was successful

# Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
%mkdir ckpts
with open("./ckpts/Weights_v1_0.pt", "wb") as file:
    file.write(response.content)

mkdir: cannot create directory ‘ckpts’: File exists


In [ ]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Weights_v1_0.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_pub2_run23_end_model.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=LineaMapper_output/2025_04_24_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224
17ESREGMAP02_Bland2021_regionB
class scores: {1: np.float64(0.5), 2: np.float64(0.5), 3: np.float64(0.5), 4: np.float64(0.5)}
demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif
minsize: 200, maxsize: 300

this script took 7.25 seconds to execute. Makes 0.00 hours.
ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.
